In [2]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Spark Job Progress Monitor already enabled


In [2]:
#Step1 - Read original Dataframe -that contains more than 1 interpretations
# result_Cohort_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Cohort_Modified_Lab")
original_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Original_Cohort_Lab_Test")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Step2 - Read Final Dataframe to test
# final_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Stacked_Cohort_lab_Test")
final_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/stacked_cohort_df_All")

In [ ]:
# original_pid_lc_dt = original_df.select("personid","labcode","New_updated_Interpretation", "servicedate").distinct()
original_pid_lc_dt = original_df.select("personid","labcode").distinct()

In [ ]:
print("original_pid_lc_dt-count", original_pid_lc_dt.count())

In [ ]:
from pyspark.sql import Window
import pyspark.sql.functions as F

# Define a window specification to partition by "personid" and "labcode"
window_spec = Window.partitionBy("personid", "labcode")

# Add a column indicating the count of records for each combination of personid and labcode
original_pid_lc_dt = original_pid_lc_dt.withColumn("record_count", F.count("*").over(window_spec))

# Filter the DataFrame to include only rows where there are more than 1 record
duplicate_records_df = original_pid_lc_dt.filter(F.col("record_count") > 1)

# Show the resulting DataFrame
duplicate_records_df.show(truncate=False)

In [ ]:
D_filtered_final_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Nonull_cohort_lab_tostack2")

In [ ]:
Stacked_pid_lc_dt = final_df.select("personid","labcode").distinct()

In [ ]:
from pyspark.sql.functions import col

# Perform a left anti-join to find distinct personid values present in original_pid_lc_dt but not in Stacked_pid_lc_dt
distinct_personid_not_in_stacked = original_pid_lc_dt.join(Stacked_pid_lc_dt, "personid", "leftanti")

# Count the distinct personid values
distinct_personid_count = distinct_personid_not_in_stacked.select("personid").distinct().count()

# Display the count of distinct personid values not present in Stacked_pid_lc_dt
print("Count of distinct personid values not present in Stacked_pid_lc_dt:", distinct_personid_count)

In [ ]:
# Collect distinct personid values to a list
distinct_personid_list = distinct_personid_not_in_stacked.select("personid").distinct().rdd.flatMap(lambda x: x).collect()

# Display the list of distinct personid values
print("Distinct personid values not present in Stacked_pid_lc_dt:")
for i in distinct_personid_list:
    print(i)

In [ ]:
from pyspark.sql.functions import col

# Convert the distinct_personid_list to a DataFrame
distinct_personid_df = spark.createDataFrame([(id,) for id in distinct_personid_list], ["personid"])

# Check if all items in distinct_personid_list are present in D_filtered_final_df
missing_personid_df = distinct_personid_df.join(D_filtered_final_df, distinct_personid_df.personid == D_filtered_final_df.personid, "left_anti")

# Count the number of items not present
missing_count = missing_personid_df.count()

# Display the items not present and the count
missing_personid_df.show()
print("Number of items not present in D_filtered_final_df:", missing_count)

In [ ]:
print(distinct_personid_df.count())

In [ ]:
from pyspark.sql.functions import col

# Perform a left anti-join to find distinct personid values present in original_pid_lc_dt but not in Stacked_pid_lc_dt
distinct_labcode_not_in_stacked = original_pid_lc_dt.join(Stacked_pid_lc_dt, "labcode", "leftanti")

# Count the distinct personid values
distinct_labcode_count = distinct_labcode_not_in_stacked.select("labcode").distinct().count()

# Display the count of distinct personid values not present in Stacked_pid_lc_dt
print("Count of distinct personid values not present in Stacked_pid_lc_dt:", distinct_labcode_count)

In [ ]:
# Collect distinct personid values to a list
distinct_lc_list = distinct_labcode_not_in_stacked.select("labcode").distinct().rdd.flatMap(lambda x: x).collect()

# Display the list of distinct personid values
print("Distinct labcode values not present in Stacked_pid_lc_dt:")
for i in distinct_lc_list:
    print(i)

In [ ]:
from pyspark.sql import functions as F
# Filter records from original_pid_lc_dt that are not present in Stacked_pid_lc_dt
unavailable_df = original_pid_lc_dt.exceptAll(Stacked_pid_lc_dt)

# Display the new unavailable_df
unavailable_df.show(50, truncate=False)

# Count the number of records in the new unavailable_df
count_unavailable = unavailable_df.count()
print("Number of records in unavailable_df:", count_unavailable)

In [ ]:
from pyspark.sql import functions as F

# Group by "personid" and "labcode", count distinct "New_updated_Interpretation"
grouped_df = unavailable_df.groupBy("personid", "labcode", "servicedate") \
    .agg(F.countDistinct("New_updated_Interpretation").alias("interpretation_count"))

# Filter out groups where the count is greater than 1
filtered_df = grouped_df.filter(grouped_df.interpretation_count > 1)

# Join filtered_df with unavailable_df to get the filtered records
filtered_records = unavailable_df.join(filtered_df, on=["personid", "labcode", "servicedate"], how="inner")

# Show the resulting DataFrame
filtered_records.show(truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Join original_df with unavailable_df based on personid and labcode
Missed_records = original_df.join(unavailable_df, ['personid', 'labcode'], 'inner') \
                     .select(original_df.columns)

# Display the final dataframe
Missed_records.show(truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Filter records with non-null New_updated_Interpretation
filtered_missed_records = Missed_records.filter(col("New_updated_Interpretation").isNotNull())

# Count the number of such records
count_filtered_records = filtered_missed_records.count()

# Display the count
print("Number of records with non-null New_updated_Interpretation:", count_filtered_records)

In [ ]:
filtered_missed_records.show(truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Filter records from original_df for matching personid and labcode from Missed_records
filtered_missing_records = original_df.join(
    Missed_records,
    (original_df['personid'] == Missed_records['personid']) & (original_df['labcode'] == Missed_records['labcode']),
    'inner'
).select(original_df['*'])

# Show the resulting DataFrame
filtered_missing_records.show(truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Check if there are any non-null records in the New_updated_Interpretation column
non_null_records_exist1 = filtered_missing_records.filter(col("New_updated_Interpretation").isNotNull()).count() > 0

if non_null_records_exist1:
    # Filter non-null records and display
    non_null_records1 = filtered_missing_records.filter(col("New_updated_Interpretation").isNotNull())
    non_null_records1.show(truncate=False)
    
    # Count the total number of non-null records
    non_null_records_count1 = non_null_records1.count()
    print("Total number of non-null records:", non_null_records_count1)
else:
    print("No non-null records found in the New_updated_Interpretation column.")

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import col, count, max, when

# Define a window partitioned by personid and labcode, ordered by servicedate
window_spec = Window.partitionBy("personid", "labcode").orderBy(col("servicedate").desc())

# Count the null records for each personid and labcode combination
null_count = Missed_records.groupBy("personid", "labcode").agg(count(when(col("New_updated_Interpretation").isNull(), True)).alias("null_count"))

# Determine the latest servicedate for each personid and labcode combination
latest_servicedate = Missed_records.withColumn("latest_servicedate", max("servicedate").over(window_spec))

# Join null_count and latest_servicedate to retain only the records meeting the conditions
Missed_records_df_null = null_count.join(latest_servicedate, ["personid", "labcode"], "inner") \
    .filter((col("null_count") > 1) & (col("servicedate") == col("latest_servicedate")) | (col("null_count") == 1)) \
    .drop("null_count", "latest_servicedate")

# Display the resulting DataFrame
Missed_records_df_null.show(truncate=False)
Distinct_Missed_records_df_null = Missed_records_df_null.distinct()
print("Total no of records - Distinct_Missed_records_df_null", Distinct_Missed_records_df_null.count())

In [ ]:
# Union the DataFrames and then apply distinct operation
stacked_df = final_df.union(Distinct_Missed_records_df_null).distinct()

print("Total no of records after following rules:",stacked_df.count())

In [ ]:
stacked_df.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cohort_Lab_ToStack")

In [ ]:
print("Count of original", original_pid_lc_dt.count())

In [ ]:
# Stacked_pid_lc_dt = stacked_df.select("personid","labcode").distinct()
print("Count of original", Stacked_pid_lc_dt.count())

In [ ]:
# Filter records in original_pid_lc_dt not present in Stacked_pid_lc_dt
missing_records = original_pid_lc_dt \
    .exceptAll(Stacked_pid_lc_dt)

# Show the resulting DataFrame
missing_records.show(truncate=False)
print("count of missing combinations in Stacked_pid_lc_dt", missing_records.count())

In [ ]:
# # Join missing_records with original_df based on matching personid and labcode
# matching_records = original_df.join(
#     missing_records,
#     (original_df['personid'] == missing_records['personid']) & (original_df['labcode'] == missing_records['labcode']),
#     'inner'
# )

# # Select all columns from original_df for the filtered records
# matching_records_filtered = matching_records.select(original_df.columns)

# # Show the resulting DataFrame
# matching_records_filtered.show(truncate=False)
# print("count of matching_records_filtered", matching_records_filtered.count())
from pyspark.sql.functions import col

# Filter records from original_df for matching personid and labcode from Missed_records
filtered_missing_records = original_df.join(
    missing_records,
    (original_df['personid'] == missing_records['personid']) & (original_df['labcode'] == missing_records['labcode']),
    'inner'
).select(original_df['*'])

# Show the resulting DataFrame
filtered_missing_records.show(truncate=False)

In [ ]:
filtered_missing_records.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/missing_records_cohort_lab")

In [ ]:
# Filter non-null New_updated_Interpretation records
non_null_interpretations = filtered_missing_records.filter(col("New_updated_Interpretation").isNotNull())

# Count the number of such records
count_non_null_interpretations = non_null_interpretations.count()

# Display the count
print("Number of non-null New_updated_Interpretation records:", count_non_null_interpretations)

In [ ]:
stacked_cohort_df_null = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/stacked_cohort_df_null")

In [ ]:
from pyspark.sql.functions import col

# Filter records from original_df for matching personid and labcode from Missed_records
filtered_missing_records1 = stacked_cohort_df_null.join(
    missing_records,
    (stacked_cohort_df_null['personid'] == missing_records['personid']) & (stacked_cohort_df_null['labcode'] == missing_records['labcode']),
    'inner'
).select(stacked_cohort_df_null['*'])

# Show the resulting DataFrame
filtered_missing_records1.show(truncate=False)

In [ ]:
original_all_dt = original_df.select("personid","labcode", "servicedate", "New_updated_Interpretation").distinct()
print("Count of original", original_all_dt.count())

In [ ]:
from pyspark.sql.functions import count

# Group by 'personid' and 'labcode' and count occurrences of New_updated_Interpretation
grouped_df = stacked_df.groupBy('personid', 'labcode').agg(count('New_updated_Interpretation').alias('interpretation_count'))

# Filter records having more than 1 New_updated_Interpretation for each labcode and personid combination
filtered_stacked_df = grouped_df.filter(grouped_df.interpretation_count > 1)

# Show the filtered DataFrame
filtered_stacked_df.show(truncate=False)
print(filtered_stacked_df.count())

In [ ]:
filtered_stacked_df = stacked_df.filter((stacked_df.personid == '96711a8c-2a6a-4db8-a3d6-8b030d69db52') & (stacked_df.labcode == '48643-1'))
filtered_stacked_df.show(truncate=False)

In [ ]:
#1
#Control Creation - Refer AFter Modifying Control with interpretations and just to do Latest SD and Mode to remove duplicates
result_Control_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Control_Modified_Lab")

In [ ]:
#2
result_Control_df.printSchema()

In [ ]:
# 3
# List of columns to drop
columns_to_drop = ["modifier", "refLowtype", "refHightype", "refLowtextvalue", "refHightextvalue", "refLowRange", "refHighRange", "value", "interpretation"]

# Dropping multiple columns
Final_Control_Lab= result_Control_df.drop(*columns_to_drop)
# Print the schema of the DataFrame to verify column names
Final_Control_Lab.printSchema()

# Check if 'servicedate' column is present in the DataFrame
if 'servicedate' in Final_Control_Lab.columns:
    print("'servicedate' column exists in the DataFrame")
else:
    print("'servicedate' column does not exist in the DataFrame")

In [ ]:
# 4- Select records with distinct values - Remove Duplicates
from pyspark.sql import functions as F

# Count the number of distinct combinations of personid, labcode, and updated_Interpretation
Final_Control_Lab = Final_Control_Lab.select("personid", "labcode", "updated_Interpretation", "servicedate").distinct()

# Show the count
print("Total number of distinct combinations of personid, labcode, and updated_Interpretation:",
      Final_Control_Lab.count())
Final_Control_Lab.show(truncate=False)

In [ ]:
# 5
from pyspark.sql.functions import col, when

accepted_values = ['Abnormal', 'Low', 'High', 'Normal', 'Positive', 'Negative']

# Filter unique updated_Interpretation values not in the accepted_values list, also empty and convert them null
unique_interpretations = Final_Control_Lab \
    .select(*[col(column) for column in Final_Control_Lab.columns], 
            when((~col("updated_Interpretation").isin(accepted_values)) | (col("updated_Interpretation") == "") | (col("updated_Interpretation").isNull()), 
                 None).otherwise(col("updated_Interpretation")).alias("New_updated_Interpretation"))

# Display unique interpretations
unique_interpretations.show(truncate=False)

In [ ]:
# Filter records
filtered_data = unique_interpretations.filter(unique_interpretations["New_updated_Interpretation"].isNull())

# Show filtered data
filtered_data.show(truncate=False)

In [ ]:
# 6
# Drop the column "updated_Interpretation"
unique_interpretations = unique_interpretations.drop("updated_Interpretation")
unique_interpretations.printSchema()

In [ ]:
# 7
from pyspark.sql.functions import to_date
# Assuming df is the DataFrame where you want to convert the servicedate column
unique_interpretations = unique_interpretations.withColumn("servicedate", to_date(unique_interpretations["servicedate"], 'yyyy-MM-dd'))

In [ ]:
# # Prioritize non-null New_updated_Interpretation values based on latest service date, breaking ties with most frequent occurrence
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, dense_rank, desc, to_timestamp
window_spec = Window.partitionBy("personid", "labcode").orderBy(desc("servicedate"))
# Filter out null values
df_non_null = unique_interpretations.filter(F.col("New_updated_Interpretation").isNotNull())
print("count of df_non_null", df_non_null.count())
# Assign rank to all non-null values based on service date to prioritize non-null values
ranked_df = df_non_null.withColumn(
    "rank", 
    dense_rank().over(window_spec)
)
# Print DataFrame schema after filtering null values
print("\nDataFrame Schema after filtering null values and ranking:")
print(ranked_df.printSchema())
print(ranked_df.show(truncate=False))
# Determine the count of non-null interpretations for each personid, labcode, and interpretation combination
interpretation_count = ranked_df.groupBy(
    "personid", "labcode", "New_updated_Interpretation"
).agg(
    F.count("*").alias("interpretation_count"),
#     F.first("servicedate").alias("max_servicedate")   # Include 'servicedate'
)

# Add a print statement to check the schema and sample rows
print("\nInterpretation Count Schema:")
print(interpretation_count.printSchema())
interpretation_count.show(5, truncate=False)
print("count of interpretation_count", interpretation_count.count())
# Determine the maximum count for each personid ,labcode and New_updated_Interpretation combination
max_count_window = Window.partitionBy("personid", "labcode")
max_count_df = interpretation_count.withColumn(
    "max_count", 
    F.max("interpretation_count").over(max_count_window)
)
# # Add a print statement to check the schema and sample rows
print("\nMax Count Schema:")
print(max_count_df.printSchema())
max_count_df.show(5, truncate=False)
print("count of max_count_df", max_count_df.count())

In [ ]:
#8
# Join ranked_df and max_count_df on common columns "personid", "labcode", and "New_updated_Interpretation"
Main_df = ranked_df.join(max_count_df, on=["personid", "labcode", "New_updated_Interpretation"], how="inner")

# Select the required columns from the joined DataFrame
result_df = Main_df.select(
    "personid",
    "labcode",
    "servicedate",
    "New_updated_Interpretation",
    "rank",
    "interpretation_count",
    "max_count"
)
print("count of result_df", result_df.count())
# Show the resulting DataFrame
result_df.show(truncate=False)

In [ ]:
#9
# Filter final_df based on rank equal to 1
filtered_final_df = result_df.filter(result_df["rank"] == 1)
print("count of filtered_final_df", filtered_final_df.count())

In [ ]:
# 10
from pyspark.sql import Window
import pyspark.sql.functions as F

# Define a window specification
window_spec = Window.partitionBy("personid", "labcode", "servicedate").orderBy(F.desc("interpretation_count"))

# Add a rank column based on interpretation_count within each group
ranked_df = filtered_final_df.withColumn("rank", F.rank().over(window_spec))

# Filter to retain only the records with rank 1 (highest interpretation_count)
most_frequent_interpretations = ranked_df.filter(F.col("rank") == 1).select(
    "personid", "labcode", "New_updated_Interpretation", "servicedate", "interpretation_count"
)

# Add a print statement to check the schema and sample rows
print("\nMost Frequent Interpretations Schema:")
print(most_frequent_interpretations.printSchema())
print("count of most_frequent_interpretations", most_frequent_interpretations.count())
most_frequent_interpretations.show(5, truncate=False)

In [ ]:
# 11
finalized_tostack_cohort_lab = most_frequent_interpretations.drop("max_count")
final_df = finalized_tostack_cohort_lab.distinct()

In [ ]:
# 12 - select random interpretation for records having same Latest SD and MODE count
from pyspark.sql import Window
import pyspark.sql.functions as F

# Define a window specification to partition by "personid" and "labcode"
window_spec = Window.partitionBy("personid", "labcode")

# Add a column indicating the count of interpretations for each combination of personid and labcode
final_df = final_df.withColumn("interpretation_count", F.count("New_updated_Interpretation").over(window_spec))

# For records where there are multiple interpretations, select one randomly
random_df = final_df.filter(F.col("interpretation_count") > 1) \
    .withColumn("random_index", F.expr("int(rand() * interpretation_count)")) \
    .groupBy("personid", "labcode", "servicedate") \
    .agg(F.first("New_updated_Interpretation").alias("New_updated_Interpretation"))

# For records where there's only one interpretation, retain them
single_df = final_df.filter(F.col("interpretation_count") <= 1) \
    .select("personid", "labcode", "New_updated_Interpretation", "servicedate")

# Union the two dataframes
filtered_final_df = single_df.union(random_df)

# Show the resulting DataFrame
filtered_final_df.show(truncate=False)

In [ ]:
# 13 - Non-Null
D_filtered_final_df = filtered_final_df.distinct()
print("count of D_filtered_final_df", D_filtered_final_df.count())

In [ ]:
##################################Control-Lab-After Filter To test############################################################
D_filtered_final_df.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/D_filtered_final_df_Control")

In [ ]:
#####Start from here Tommo #######################
D_filtered_final_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/D_filtered_final_df_Control")

In [ ]:
unique_interpretations_Pid_LC = unique_interpretations.select("personid","labcode").distinct()
df_non_null_Pid_LC = df_non_null.select("personid", "labcode").distinct()

In [ ]:
# We determined individual personid and individual labcode but didn't determine the combination of missing pid and lc
# Extract distinct personid and labcode from unique_interpretations_Pid_LC
distinct_personid_labs_unique_interpretations = unique_interpretations_Pid_LC.select("personid").distinct()
distinct_labcode_labs_unique_interpretations = unique_interpretations_Pid_LC.select("labcode").distinct()
print("count of dt personid from original df", distinct_personid_labs_unique_interpretations.count())
print("count of dt labcode from originall df", distinct_labcode_labs_unique_interpretations.count())
# Extract distinct personid and labcode from df_non_null
distinct_personid_labs_df_non_null = df_non_null_Pid_LC.select("personid").distinct()
distinct_labcode_labs_df_non_null = df_non_null_Pid_LC.select("labcode").distinct()
print("count of dt personid from non-null df", distinct_personid_labs_df_non_null.count())
print("count of dt  labcode from non-null df", distinct_labcode_labs_df_non_null.count())
# Find missing distinct personid and labcode
missing_personid = distinct_personid_labs_unique_interpretations.subtract(distinct_personid_labs_df_non_null)
missing_labcode = distinct_labcode_labs_unique_interpretations.subtract(distinct_labcode_labs_df_non_null)

# Print the number of missing distinct personid and labcode
if missing_personid.count() > 0:
    print("Number of missing distinct personid:", missing_personid.count())
    missing_personid.show(truncate=False)
else:
    print("No missing distinct personid found.")

if missing_labcode.count() > 0:
    print("Number of missing distinct labcode:", missing_labcode.count())
    missing_labcode.show(truncate=False)
else:
    print("No missing distinct labcode found.")

In [ ]:
unique_interpretations.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/unique_interpretations_control")

In [ ]:
# Perform a left anti join to find personids present in Missed_null_records but not in missing_personid
missing_personids = Missed_null_records.join(
    missing_personid,
    Missed_null_records['personid'] == missing_personid['personid'],
    'left_anti'
)

# Count the number of such records
count_missing_personids = missing_personids.count()

# Display the records and count
if count_missing_personids > 0:
    print("Person IDs present in Missed_null_records but not in missing_personid:")
    missing_personids.show()
    print("Number of such records:", count_missing_personids)
else:
    print("No missing person IDs found.")

In [ ]:
from pyspark.sql.functions import col
filtered_missing_labcodes = unique_interpretations.join(
    missing_labcode,
    unique_interpretations['labcode'] == missing_labcode['labcode'],
    'inner'
).select(unique_interpretations['*'])
filtered_missing_labcodes.show(truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Check if there are any non-null records in the New_updated_Interpretation column
non_null_records_exist = filtered_missing_labcodes.filter(col("New_updated_Interpretation").isNotNull()).count() > 0

if non_null_records_exist:
    # Filter non-null records and display
    non_null_records = filtered_missing_labcodes.filter(col("New_updated_Interpretation").isNotNull())
    non_null_records.show(truncate=False)
    
    # Count the total number of non-null records
    non_null_records_count = non_null_records.count()
    print("Total number of non-null records:", non_null_records_count)
else:
    print("No non-null records found in the New_updated_Interpretation column.")

In [ ]:
from pyspark.sql.functions import col

# Assuming unique_interpretations has a column named 'labcode'
filtered_missing_personid = unique_interpretations.join(
    missing_personid,
    unique_interpretations['personid'] == missing_personid['personid'],
    'inner'
).select(unique_interpretations['*'])
filtered_missing_personid.show(truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Check if there are any non-null records in the New_updated_Interpretation column
non_null_records_exist1 = filtered_missing_personid.filter(col("New_updated_Interpretation").isNotNull()).count() > 0

if non_null_records_exist1:
    # Filter non-null records and display
    non_null_records1 = filtered_missing_personid.filter(col("New_updated_Interpretation").isNotNull())
    non_null_records1.show(truncate=False)
    
    # Count the total number of non-null records
    non_null_records_count1 = non_null_records1.count()
    print("Total number of non-null records:", non_null_records_count1)
else:
    print("No non-null records found in the New_updated_Interpretation column.")

In [ ]:
from pyspark.sql import functions as F

# Join distinct_df and missing_personid on personid
joined_df = distinct_df.join(missing_personid, distinct_df.personid == missing_personid.personid, "inner")

# Filter the joined DataFrame to keep only the records where personid is present in both DataFrames
filtered_df = joined_df.select(distinct_df.personid).distinct()

# Count the distinct personids
count_distinct_personids = filtered_df.count()

# Display the count
print("Count of distinct personids present in both distinct_df and missing_personid:", count_distinct_personids)

In [ ]:
from pyspark.sql import functions as F

# Join distinct_df and missing_personid on personid
joined_df = distinct_df.join(missing_labcode, distinct_df.labcode == missing_labcode.labcode, "inner")

# Filter the joined DataFrame to keep only the records where personid is present in both DataFrames
filtered_df = joined_df.select(distinct_df.labcode).distinct()

# Count the distinct personids
count_distinct_labcode = filtered_df.count()

# Display the count
print("Count of distinct personids present in both distinct_df and missing_labcode:", count_distinct_labcode)

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import col, count, max, when

# Define a window partitioned by personid and labcode, ordered by servicedate
window_spec = Window.partitionBy("personid", "labcode").orderBy(col("servicedate").desc())

# Count the null records for each personid and labcode combination
null_count = filtered_missing_labcodes.groupBy("personid", "labcode").agg(count(when(col("New_updated_Interpretation").isNull(), True)).alias("null_count"))

# Determine the latest servicedate for each personid and labcode combination
latest_servicedate = filtered_missing_labcodes.withColumn("latest_servicedate", max("servicedate").over(window_spec))

# Join null_count and latest_servicedate to retain only the records meeting the conditions
final_df_null = null_count.join(latest_servicedate, ["personid", "labcode"], "inner") \
    .filter((col("null_count") > 1) & (col("servicedate") == col("latest_servicedate")) | (col("null_count") == 1)) \
    .drop("null_count", "latest_servicedate")

# Display the resulting DataFrame
final_df_null.show(truncate=False)
Distinct_filtered_missing_labcodes = filtered_missing_labcodes.distinct()
Distinct_final_df_null = final_df_null.distinct()
print("Total no of original records", filtered_missing_labcodes.count())
print("Total no of distinct original records", Distinct_filtered_missing_labcodes.count())
print("Total no of records", final_df_null.count())
print("Total no of records", Distinct_final_df_null.count())

In [ ]:
from pyspark.sql import functions as F

# Count the occurrences of each combination of "personid" and "labcode"
combination_counts = Distinct_final_df_null.groupBy("personid", "labcode").count()

# Filter out combinations with counts greater than 1
unique_combinations = combination_counts.filter(F.col("count") > 1)

# Show the unique combinations
unique_combinations.show(truncate=False)

# Count of unique combinations
print("Count of unique combinations:", unique_combinations.count())
print("count of Distinct_final_df_null for pid and lc:", Distinct_final_df_null.select("personid","labcode").distinct().count())

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import col, count, max, when

# Define a window partitioned by personid and labcode, ordered by servicedate
window_spec = Window.partitionBy("personid", "labcode").orderBy(col("servicedate").desc())

# Count the null records for each personid and labcode combination
null_count = filtered_missing_personid.groupBy("personid", "labcode").agg(count(when(col("New_updated_Interpretation").isNull(), True)).alias("null_count"))

# Determine the latest servicedate for each personid and labcode combination
latest_servicedate = filtered_missing_personid.withColumn("latest_servicedate", max("servicedate").over(window_spec))

# Join null_count and latest_servicedate to retain only the records meeting the conditions
final_df_null1 = null_count.join(latest_servicedate, ["personid", "labcode"], "inner") \
    .filter((col("null_count") > 1) & (col("servicedate") == col("latest_servicedate")) | (col("null_count") == 1)) \
    .drop("null_count", "latest_servicedate")

# Display the resulting DataFrame
final_df_null1.show(truncate=False)
Distinct_filtered_missing_personid = filtered_missing_personid.distinct()
Distinct_final_df_null1 = final_df_null1.distinct()
print("Total no of original records", filtered_missing_personid.count())
print("Total no of distinct original records", Distinct_filtered_missing_personid.count())
print("Total no of records", final_df_null1.count())
print("Total no of records", Distinct_final_df_null1.count())

In [ ]:
from pyspark.sql.functions import col

# Filter distinct combinations of personid and labcode
distinct_person_lab = filtered_missing_labcodes.select("personid", "labcode").distinct()

# Count the number of such distinct combinations
count_records = distinct_person_lab.count()

# Display the count
print("Number of distinct personid and labcode combinations:", count_records)

In [ ]:
from pyspark.sql import functions as F

# Count the occurrences of each combination of "personid" and "labcode"
combination_counts = Distinct_final_df_null1.groupBy("personid", "labcode").count()

# Filter out combinations with counts greater than 1
unique_combinations = combination_counts.filter(F.col("count") > 1)

# Show the unique combinations
unique_combinations.show(truncate=False)

# Count of unique combinations
print("Count of unique combinations:", unique_combinations.count())
print("count of Distinct_final_df_null for pid and lc:", Distinct_final_df_null1.select("personid","labcode").distinct().count())

In [ ]:
# Distinct_final_df1 = Distinct_final_df.drop("rank")
# Distinct_final_df1 = Distinct_final_df1.withColumnRenamed("max_servicedate", "servicedate")
stacked_controldf_null = Distinct_final_df_null.union(Distinct_final_df_null1).distinct()

In [ ]:
##################################write-stacked_control_df_null ############################################################
stacked_controldf_null.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/stacked_control_df_null")

In [ ]:
# Union the DataFrames and then apply distinct operation
# stacked_control_df_All = D_filtered_final_df.union(Distinct_final_df_null).union(Distinct_final_df_null1)
stacked_control_df_All = D_filtered_final_df.union(stacked_controldf_null)
# Keep only distinct values
stacked_control_df_All = stacked_control_df_All.distinct()

# Show the resulting DataFrame
stacked_control_df_All.show(truncate=False)

# Count the number of rows in the final DataFrame
print("Count of rows in the stacked_cohort_df:", stacked_control_df_All.count())

In [ ]:
##################################Control-Lab-After Filter To test############################################################
stacked_control_df_All.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/stacked_control_df_All")

In [ ]:
#Original Control Lab Test
original_CT_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/unique_interpretations_control")

In [ ]:
# Control -Modified with MODE and Latest SD
final_CT_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/stacked_control_df_All")

In [ ]:
original_pid_lc_CT_dt = original_CT_df.select("personid","labcode").distinct()
print("original_pid_lc_dt-count", original_pid_lc_CT_dt.count())

In [ ]:
Stacked_pid_lc_CT_dt = final_CT_df.select("personid","labcode").distinct()
print("original_pid_lc_dt-count", Stacked_pid_lc_CT_dt.count())

In [ ]:
from pyspark.sql.functions import col

# Perform a left anti-join to find distinct personid values present in original_pid_lc_dt but not in Stacked_pid_lc_dt
distinct_personid_not_in_stacked_CTL = original_pid_lc_CT_dt.join(Stacked_pid_lc_CT_dt, "personid", "leftanti")

# Count the distinct personid values
distinct_CTL_personid_count = distinct_personid_not_in_stacked_CTL.select("personid").distinct().count()

# Display the count of distinct personid values not present in Stacked_pid_lc_dt
print("Count of distinct personid values not present in Stacked_pid_lc_CT_dt:", distinct_CTL_personid_count)

In [ ]:
from pyspark.sql.functions import col

# Perform a left anti-join to find distinct personid values present in original_pid_lc_dt but not in Stacked_pid_lc_dt
distinct_labcode_not_in_stacked_CTL = original_pid_lc_CT_dt.join(Stacked_pid_lc_CT_dt, "labcode", "leftanti")

# Count the distinct personid values
distinct_CTL_labcode_count = distinct_labcode_not_in_stacked_CTL.select("labcode").distinct().count()

# Display the count of distinct personid values not present in Stacked_pid_lc_CT_dt
print("Count of distinct personid values not present in Stacked_pid_lc_CT_dt:", distinct_CTL_labcode_count)

In [ ]:
from pyspark.sql import functions as F
# Filter records from original_pid_lc_dt that are not present in Stacked_pid_lc_dt
unavailable_CT_df = original_pid_lc_CT_dt.exceptAll(Stacked_pid_lc_CT_dt)

# Display the new unavailable_df
unavailable_CT_df.show(50, truncate=False)

# Count the number of records in the new count_CT_unavailable
count_CT_unavailable = unavailable_CT_df.count()
print("Number of records in unavailable_df:", count_CT_unavailable)

In [ ]:
from pyspark.sql.functions import col

# Join original_df with unavailable_df based on personid and labcode
Missed_CT_records = original_CT_df.join(unavailable_CT_df, ['personid', 'labcode'], 'inner') \
                     .select(original_CT_df.columns)

# Display the final dataframe
Missed_CT_records.show(truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Filter records with non-null New_updated_Interpretation
filtered_missed_records_CT = Missed_CT_records.filter(col("New_updated_Interpretation").isNotNull())

# Count the number of such records
count_filtered_records_CT = filtered_missed_records_CT.count()

# Display the count
print("Number of records with non-null New_updated_Interpretation:", count_filtered_records_CT)

In [ ]:
##################################Control-Lab-MissedRecord############################################################
Missed_CT_records.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/filtered_missed_records_CT")

In [ ]:
Missed_Control_Records = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/filtered_missed_records_CT")

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import col, count, max, when

# Define a window partitioned by personid and labcode, ordered by servicedate
window_spec = Window.partitionBy("personid", "labcode").orderBy(col("servicedate").desc())

# Count the null records for each personid and labcode combination
null_count_Missed = Missed_Control_Records.groupBy("personid", "labcode").agg(count(when(col("New_updated_Interpretation").isNull(), True)).alias("null_count"))

# Determine the latest servicedate for each personid and labcode combination
latest_servicedate = Missed_Control_Records.withColumn("latest_servicedate", max("servicedate").over(window_spec))

# Join null_count and latest_servicedate to retain only the records meeting the conditions
Missed_CL_records_df_null = null_count_Missed.join(latest_servicedate, ["personid", "labcode"], "inner") \
    .filter((col("null_count") > 1) & (col("servicedate") == col("latest_servicedate")) | (col("null_count") == 1)) \
    .drop("null_count", "latest_servicedate")

# Display the resulting DataFrame
Missed_CL_records_df_null.show(truncate=False)
Distinct_Missed_CL_records_df_null = Missed_CL_records_df_null.distinct()
print("Total no of records - Distinct_Missed_CL_records_df_null", Distinct_Missed_CL_records_df_null.count())

In [ ]:
# Union the DataFrames and then apply distinct operation
stacked_df_Full_Control1 = final_CT_df.union(Distinct_Missed_CL_records_df_null).distinct()

print("Total no of records after following rules:",stacked_df_Full_Control1.count())

In [ ]:
stacked_df_Full_Control1.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Control_Lab_ToStack")

In [9]:
Final_Cohort_Lab_toStack = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cohort_Lab_ToStack")

In [4]:
Final_Control_Lab_toStack = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Control_Lab_ToStack")

In [7]:
Final_Cohort_Lab_toStack.printSchema()
Final_Control_Lab_toStack.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [5]:
from pyspark.sql.functions import col, to_date

# Filter records with non-null servicedate and non-date servicedate
filtered_df = Final_Cohort_Lab_toStack.filter(col("servicedate").isNotNull() & ~(to_date(col("servicedate"), "yyyy-MM-dd").isNotNull()))

# Show the filtered DataFrame
filtered_df.show(truncate=False)

+------------------------------------+---------+--------------------------+-----------+
|personid                            |labcode  |New_updated_Interpretation|servicedate|
+------------------------------------+---------+--------------------------+-----------+
|a100ea3d-e166-45a3-9dfa-81b9de6e6c33|1975-2   |2016-10-09                |Low        |
|2f9a6597-326b-4e73-a403-9b0a82d23f56|2714-4   |2018-01-28                |Low        |
|038f0837-fc56-48f3-bfe8-a38769112788|57617002 |2019-08-18                |Abnormal   |
|7b0bb4aa-d9fb-466f-9207-18f341d9bfc4|119297000|2019-12-18                |Normal     |
|c88ee4e1-2c1c-4423-9750-23b04e238921|122575003|2021-05-15                |High       |
|289605e9-2253-45c8-977f-eadeaabdb0be|5799-2   |2021-06-27                |Abnormal   |
|ae70a623-a868-4465-b557-0391e76af931|14979-9  |2017-04-13                |High       |
|23d83630-2984-440a-b687-3a5f412a7c66|785-6    |2019-09-01                |Normal     |
|3facf3ba-a209-4eaf-8551-eaa66d2

In [13]:
# from pyspark.sql.functions import when, col, to_date

# # Create a new column to check if New_updated_Interpretation contains a date
# Final_Cohort_Lab_toStack_Cleaned = Final_Cohort_Lab_toStack.withColumn("interpretation_is_date", to_date(col("New_updated_Interpretation"), "yyyy-MM-dd").isNotNull())

# # Create a new column to check if servicedate contains a non-date value
# Final_Cohort_Lab_toStack_Cleaned = Final_Cohort_Lab_toStack_Cleaned.withColumn("servicedate_is_not_date", ~(to_date(col("servicedate"), "yyyy-MM-dd").isNotNull()))

# # Swap values where necessary
# Final_Cohort_Lab_toStack_Cleaned = Final_Cohort_Lab_toStack_Cleaned.withColumn("New_updated_Interpretation", 
#                    when(col("interpretation_is_date") & col("servicedate_is_not_date"), col("servicedate"))
#                    .otherwise(col("New_updated_Interpretation")))

# Final_Cohort_Lab_toStack_Cleaned = Final_Cohort_Lab_toStack_Cleaned.withColumn("servicedate", 
#                    when(col("interpretation_is_date") & col("servicedate_is_not_date"), col("New_updated_Interpretation"))
#                    .otherwise(col("servicedate")))

# # Drop the temporary columns
# Final_Cohort_Lab_toStack_Cleaned = Final_Cohort_Lab_toStack_Cleaned.drop("interpretation_is_date", "servicedate_is_not_date")

# # Show the DataFrame after swapping values
# Final_Cohort_Lab_toStack_Cleaned.show(truncate=False)
from pyspark.sql.functions import when, col, to_date

# Create a new column to check if New_updated_Interpretation contains a date
Final_Cohort_Lab_toStack_Cleaned = Final_Cohort_Lab_toStack.withColumn("interpretation_is_date", to_date(col("New_updated_Interpretation"), "yyyy-MM-dd").isNotNull())

# Create a new column to check if servicedate contains a non-date value
Final_Cohort_Lab_toStack_Cleaned = Final_Cohort_Lab_toStack_Cleaned.withColumn("servicedate_is_not_date", ~(to_date(col("servicedate"), "yyyy-MM-dd").isNotNull()))

# Swap values where necessary
Final_Cohort_Lab_toStack_Cleaned = Final_Cohort_Lab_toStack_Cleaned.withColumn("temp", 
                   when(col("interpretation_is_date") & col("servicedate_is_not_date"), col("servicedate"))
                   .when(~col("interpretation_is_date") & ~col("servicedate_is_not_date"), col("New_updated_Interpretation"))
                   .otherwise(None))

Final_Cohort_Lab_toStack_Cleaned = Final_Cohort_Lab_toStack_Cleaned.withColumn("servicedate", 
                   when(col("interpretation_is_date") & col("servicedate_is_not_date"), col("New_updated_Interpretation"))
                   .otherwise(col("servicedate")))

Final_Cohort_Lab_toStack_Cleaned = Final_Cohort_Lab_toStack_Cleaned.withColumn("New_updated_Interpretation", 
                   when(col("interpretation_is_date") & col("servicedate_is_not_date"), col("temp"))
                   .otherwise(col("New_updated_Interpretation")))

# Drop the temporary columns
Final_Cohort_Lab_toStack_Cleaned = Final_Cohort_Lab_toStack_Cleaned.drop("interpretation_is_date", "servicedate_is_not_date", "temp")

# Show the DataFrame after swapping values
Final_Cohort_Lab_toStack_Cleaned.show(truncate=False)

+------------------------------------+-------+--------------------------+-----------+
|personid                            |labcode|New_updated_Interpretation|servicedate|
+------------------------------------+-------+--------------------------+-----------+
|05b46377-fece-48be-a1f5-bad5bb7842fd|770-8  |Normal                    |2018-05-11 |
|07ea477b-5291-44d3-8b3a-b137abd708d5|718-7  |Normal                    |2014-12-24 |
|09387be8-5f66-4def-b75e-08290e155a43|4544-3 |Low                       |2020-10-27 |
|09d35d50-fd6f-4a64-85b1-a372fd199277|2498-4 |Normal                    |2019-06-10 |
|0a160575-7be7-48a3-8413-70d84b22108c|751-8  |High                      |2021-03-02 |
|0a2a84b8-b4bc-4aba-998b-8381748f0481|41276-7|Normal                    |2017-02-11 |
|0acb2551-6ec8-4d71-9994-70fbbf48676b|5778-6 |Abnormal                  |2020-12-16 |
|0d320e8f-4dca-49b9-a3b1-1dc46fa211f1|14627-4|Normal                    |2018-11-16 |
|0df15f4d-123c-4287-bb90-fdbf9be29747|5905-5 |Normal  

In [14]:
from pyspark.sql.functions import col, to_date

# Filter records with non-null servicedate and non-date servicedate
filtered_df = Final_Cohort_Lab_toStack_Cleaned.filter(col("servicedate").isNotNull() & ~(to_date(col("servicedate"), "yyyy-MM-dd").isNotNull()))

# Show the filtered DataFrame
filtered_df.show(truncate=False)

+--------+-------+--------------------------+-----------+
|personid|labcode|New_updated_Interpretation|servicedate|
+--------+-------+--------------------------+-----------+
+--------+-------+--------------------------+-----------+



In [23]:
print(Final_Cohort_Lab_toStack.count())
print(Final_Cohort_Lab_toStack_Cleaned.count())

6273097
6273097


In [21]:
Final_Cohort_Lab_toStack_Cleaned.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cohort_Lab_toStack_Cleaned")

In [6]:
from pyspark.sql.functions import col, to_date

# Filter records with non-null servicedate and non-date servicedate
filtered_df = Final_Control_Lab_toStack.filter(col("servicedate").isNotNull() & ~(to_date(col("servicedate"), "yyyy-MM-dd").isNotNull()))

# Show the filtered DataFrame
filtered_df.show(truncate=False)

+------------------------------------+---------+--------------------------+-----------+
|personid                            |labcode  |New_updated_Interpretation|servicedate|
+------------------------------------+---------+--------------------------+-----------+
|00263306-93b7-4adb-84a5-3a8e3fde5672|80348-6  |2022-06-30                |Abnormal   |
|01081071-34c3-426c-a0cb-92e5be171805|10466-1  |2021-06-23                |Normal     |
|01bd019b-f4a4-42e9-88b2-da4bbc11d630|119297000|2021-09-18                |Low        |
|02e3295b-ee42-4c0f-aea3-254f05ebe8c8|61152-5  |2021-04-18                |Normal     |
|0366a590-d2fd-495f-abed-fede13b5d1fc|41653-7  |2016-01-27                |High       |
|0449ca1f-b4ba-4f76-af44-2c51bcdcd943|57617002 |2021-03-23                |Normal     |
|04608f16-e90e-4362-9e1d-f4d0606e5f9e|2823-3   |2016-11-06                |Low        |
|04e9b7c8-2286-43fb-af5d-581bb80776fa|122575003|2019-02-17                |Low        |
|055b26a0-a284-44dd-a38d-f160c5f

In [18]:
from pyspark.sql.functions import col, to_date

# Assuming spark is your SparkSession

# Filter records based on personid, labcode, and servicedate conditions
filtered_df = Final_Control_Lab_toStack.filter(
    (col("personid") == "00263306-93b7-4adb-84a5-3a8e3fde5672") &
    (col("labcode") == "80348-6") &
    ~(to_date(col("servicedate"), "yyyy-MM-dd").isNotNull())
)

# Show the filtered DataFrame
filtered_df.show(truncate=False)

+------------------------------------+-------+--------------------------+-----------+
|personid                            |labcode|New_updated_Interpretation|servicedate|
+------------------------------------+-------+--------------------------+-----------+
|00263306-93b7-4adb-84a5-3a8e3fde5672|80348-6|2022-06-30                |Abnormal   |
+------------------------------------+-------+--------------------------+-----------+



In [15]:
from pyspark.sql.functions import when, col, to_date

# Create a new column to check if New_updated_Interpretation contains a date
Final_Control_Lab_toStack_Cleaned = Final_Control_Lab_toStack.withColumn("interpretation_is_date", to_date(col("New_updated_Interpretation"), "yyyy-MM-dd").isNotNull())

# Create a new column to check if servicedate contains a non-date value
Final_Control_Lab_toStack_Cleaned = Final_Control_Lab_toStack_Cleaned.withColumn("servicedate_is_not_date", ~(to_date(col("servicedate"), "yyyy-MM-dd").isNotNull()))

# Swap values where necessary
Final_Control_Lab_toStack_Cleaned = Final_Control_Lab_toStack_Cleaned.withColumn("temp", 
                   when(col("interpretation_is_date") & col("servicedate_is_not_date"), col("servicedate"))
                   .when(~col("interpretation_is_date") & ~col("servicedate_is_not_date"), col("New_updated_Interpretation"))
                   .otherwise(None))

Final_Control_Lab_toStack_Cleaned = Final_Control_Lab_toStack_Cleaned.withColumn("servicedate", 
                   when(col("interpretation_is_date") & col("servicedate_is_not_date"), col("New_updated_Interpretation"))
                   .otherwise(col("servicedate")))

Final_Control_Lab_toStack_Cleaned = Final_Control_Lab_toStack_Cleaned.withColumn("New_updated_Interpretation", 
                   when(col("interpretation_is_date") & col("servicedate_is_not_date"), col("temp"))
                   .otherwise(col("New_updated_Interpretation")))

# Drop the temporary columns
Final_Control_Lab_toStack_Cleaned = Final_Control_Lab_toStack_Cleaned.drop("interpretation_is_date", "servicedate_is_not_date", "temp")

# Show the DataFrame after swapping values
Final_Control_Lab_toStack_Cleaned.show(truncate=False)

+------------------------------------+-------+--------------------------+-----------+
|personid                            |labcode|New_updated_Interpretation|servicedate|
+------------------------------------+-------+--------------------------+-----------+
|000058c9-4684-4a8a-9913-6a6ba01f8208|53797-7|null                      |2021-10-08 |
|000086b2-3048-43ce-a370-25dfde6b7fa7|3094-0 |Normal                    |2017-10-21 |
|000151d8-40ee-46d4-860a-a757a06ff3b7|5905-5 |Normal                    |2021-09-18 |
|00019068-80ee-4e89-884e-72a73e77eae7|788-0  |Normal                    |2017-12-20 |
|000290a4-6823-491b-9145-518a5812281f|5902-2 |Normal                    |2022-03-22 |
|0005761f-eaa6-43f0-84f4-f023f0d682dd|1975-2 |Normal                    |2021-07-14 |
|0005761f-eaa6-43f0-84f4-f023f0d682dd|4544-3 |Normal                    |2021-07-14 |
|000605e3-f575-4d22-93b7-e2343af7d910|33037-3|Normal                    |2020-06-24 |
|00060bca-a396-410d-99c6-44ed33e116f0|5902-2 |High    

In [16]:
from pyspark.sql.functions import col, to_date

# Filter records with non-null servicedate and non-date servicedate
filtered_df = Final_Control_Lab_toStack_Cleaned.filter(col("servicedate").isNotNull() & ~(to_date(col("servicedate"), "yyyy-MM-dd").isNotNull()))

# Show the filtered DataFrame
filtered_df.show(truncate=False)

+--------+-------+--------------------------+-----------+
|personid|labcode|New_updated_Interpretation|servicedate|
+--------+-------+--------------------------+-----------+
+--------+-------+--------------------------+-----------+



In [20]:
from pyspark.sql.functions import col, to_date

# Assuming spark is your SparkSession

# Filter records based on personid, labcode, and servicedate conditions
filtered_df = Final_Control_Lab_toStack_Cleaned.filter(
    (col("personid") == "00263306-93b7-4adb-84a5-3a8e3fde5672") &
    (col("labcode") == "80348-6") &
    (col("New_updated_Interpretation") == "Abnormal")
#     ~(to_date(col("servicedate"), "yyyy-MM-dd").isNotNull())
)

# Show the filtered DataFrame
filtered_df.show(truncate=False)

+------------------------------------+-------+--------------------------+-----------+
|personid                            |labcode|New_updated_Interpretation|servicedate|
+------------------------------------+-------+--------------------------+-----------+
|00263306-93b7-4adb-84a5-3a8e3fde5672|80348-6|Abnormal                  |2022-06-30 |
+------------------------------------+-------+--------------------------+-----------+



In [22]:
Final_Control_Lab_toStack_Cleaned.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Control_Lab_toStack_Cleaned")

In [24]:
print(Final_Control_Lab_toStack.count())
print(Final_Control_Lab_toStack_Cleaned.count())

16273634
16273634


In [8]:
Cohort_Lab_CT = Final_Cohort_Lab_toStack.count()
Control_Lab_CT = Final_Control_Lab_toStack.count()

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
print("Cohort_Lab_CT:", Cohort_Lab_CT)
print("Control_Lab_CT:", Control_Lab_CT)

▸,:,


Cohort_Lab_CT: 6273097
Control_Lab_CT: 16273634


In [40]:
from pyspark.sql.functions import col, count, when

# Calculate the total count of personid records for each labcode
labcode_cohort_counts = Final_Cohort_Lab_toStack_Cleaned.groupBy("labcode") \
    .agg(count("personid").alias("total_count"))

# Calculate the count of personid records where New_updated_Interpretation is null for each labcode
null_interpretation_cohort_counts = Final_Cohort_Lab_toStack_Cleaned.groupBy("labcode") \
    .agg(count(when(col("New_updated_Interpretation").isNull(), True)).alias("null_interpretation_count"))

# Join the two DataFrames to calculate the percentage of null interpretations
joined_cohort_counts = labcode_cohort_counts.join(null_interpretation_cohort_counts, "labcode", "inner")

# Calculate the percentage of null interpretations
joined_cohort_counts = joined_cohort_counts.withColumn("null_interpretation_percentage", 
                                         (col("null_interpretation_count") / col("total_count")) * 100)

# Filter labcodes where null interpretation percentage is greater than 90%
filtered_Cohort_labcodes = joined_cohort_counts.filter(col("null_interpretation_percentage") > 90)

# Count the number of such unique labcodes
count_filtered_Cohort_labcodes = filtered_Cohort_labcodes.select("labcode").distinct().count()

# Display the count of unique labcodes
print("Number of unique labcodes with more than 90% personid having null New_updated_Interpretation for cohort group:", count_filtered_Cohort_labcodes)

Number of unique labcodes with more than 90% personid having null New_updated_Interpretation: 3515


In [30]:
# Assuming spark is your SparkSession

# Count the total number of unique labcodes
unique_Cohort_labcode_count = Final_Cohort_Lab_toStack_Cleaned.select("labcode").distinct().count()

# Print the total number of unique labcodes
print("Total number of unique labcodes:", unique_Cohort_labcode_count)

Total number of unique labcodes: 8836


In [41]:
from pyspark.sql.functions import col, count, when

# Calculate the total count of personid records for each labcode
labcode_control_counts = Final_Control_Lab_toStack_Cleaned.groupBy("labcode") \
    .agg(count("personid").alias("total_count"))

# Calculate the count of personid records where New_updated_Interpretation is null for each labcode
null_interpretation_control_counts = Final_Control_Lab_toStack_Cleaned.groupBy("labcode") \
    .agg(count(when(col("New_updated_Interpretation").isNull(), True)).alias("null_interpretation_count"))

# Join the two DataFrames to calculate the percentage of null interpretations
joined_control_counts = labcode_control_counts.join(null_interpretation_control_counts, "labcode", "inner")

# Calculate the percentage of null interpretations
joined_control_counts = joined_control_counts.withColumn("null_interpretation_percentage", 
                                         (col("null_interpretation_count") / col("total_count")) * 100)

# Filter labcodes where null interpretation percentage is greater than 90%
filtered_control_labcodes = joined_control_counts.filter(col("null_interpretation_percentage") > 90)

# Count the number of such unique labcodes
count_filtered_control_labcodes = filtered_control_labcodes.select("labcode").distinct().count()

# Display the count of unique labcodes
print("Number of unique labcodes with more than 90% personid having null New_updated_Interpretation for control group:", count_filtered_control_labcodes)

Number of unique labcodes with more than 90% personid having null New_updated_Interpretation for control group: 4300


In [36]:
filtered_labcodes.printSchema()

root
 |-- labcode: string (nullable = true)
 |-- total_count: long (nullable = false)
 |-- null_interpretation_count: long (nullable = false)
 |-- null_interpretation_percentage: double (nullable = true)



In [45]:
cohort = filtered_Cohort_labcodes.select("labcode").distinct()
control = filtered_control_labcodes.select("labcode").distinct()
Total_labcode_Eliminate = cohort.union(control).distinct().count()
print("Total_labcode_Eliminate from 10825", Total_labcode_Eliminate)

Total_labcode_Eliminate from 10825 5256


In [37]:
filtered_labcodes_cohort_count = filtered_labcodes.select("labcode").distinct().count()

In [38]:
print(filtered_labcodes_cohort_count)

4500


In [29]:
# Assuming spark is your SparkSession

# Count the total number of unique labcodes
unique_CTL_labcode_count = Final_Control_Lab_toStack_Cleaned.select("labcode").distinct().count()

# Print the total number of unique labcodes
print("Total number of unique labcodes:", unique_CTL_labcode_count)

Total number of unique labcodes: 10274


In [31]:
from pyspark.sql.functions import col

# Concatenate the two DataFrames
stacked_df = Final_Cohort_Lab_toStack_Cleaned.union(Final_Control_Lab_toStack_Cleaned)

# Count the total number of distinct labcodes
distinct_labcode_count = stacked_df.select("labcode").distinct().count()

# Print the total number of distinct labcodes
print("Total number of distinct labcodes:", distinct_labcode_count)

Total number of distinct labcodes: 10825


In [33]:
stacked_df = stacked_df.distinct()
stacked_df.printSchema()

root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)



In [34]:
from pyspark.sql import functions as F

# Group by personid and labcode, and count distinct New_updated_Interpretation
grouped_df = stacked_df.groupBy("personid", "labcode") \
                        .agg(F.countDistinct("New_updated_Interpretation").alias("interpretation_count"))

# Filter records where interpretation count is greater than 1
filtered_df = grouped_df.filter("interpretation_count > 1")

# Show the filtered DataFrame
filtered_df.show(truncate=False)

+--------+-------+--------------------+
|personid|labcode|interpretation_count|
+--------+-------+--------------------+
+--------+-------+--------------------+



In [35]:
from pyspark.sql.functions import col, count, when

# Calculate the total count of personid records for each labcode
labcode_counts = stacked_df.groupBy("labcode") \
    .agg(count("personid").alias("total_count"))

# Calculate the count of personid records where New_updated_Interpretation is null for each labcode
null_interpretation_counts = stacked_df.groupBy("labcode") \
    .agg(count(when(col("New_updated_Interpretation").isNull(), True)).alias("null_interpretation_count"))

# Join the two DataFrames to calculate the percentage of null interpretations
joined_counts = labcode_counts.join(null_interpretation_counts, "labcode", "inner")

# Calculate the percentage of null interpretations
joined_counts = joined_counts.withColumn("null_interpretation_percentage", 
                                         (col("null_interpretation_count") / col("total_count")) * 100)

# Filter labcodes where null interpretation percentage is greater than 90%
filtered_labcodes = joined_counts.filter(col("null_interpretation_percentage") > 90)

# Count the number of such unique labcodes
count_filtered_labcodes = filtered_labcodes.count()

# Display the count of unique labcodes
print("Number of unique labcodes with more than 90% null New_updated_Interpretation:", count_filtered_labcodes)

Number of unique labcodes with more than 90% null New_updated_Interpretation: 4500


In [26]:
from pyspark.sql.functions import col, count, when

# Calculate the total count of personid records for each labcode
labcode_counts = Final_Control_Lab_toStack_Cleaned.groupBy("labcode") \
    .agg(count("personid").alias("total_count"))

# Calculate the count of personid records where New_updated_Interpretation is null for each labcode
null_interpretation_counts = Final_Control_Lab_toStack_Cleaned.groupBy("labcode") \
    .agg(count(when(col("New_updated_Interpretation").isNull(), True)).alias("null_interpretation_count"))

# Join the two DataFrames to calculate the percentage of null interpretations
joined_counts = labcode_counts.join(null_interpretation_counts, "labcode", "inner")

# Calculate the percentage of null interpretations
joined_counts = joined_counts.withColumn("null_interpretation_percentage", 
                                         (col("null_interpretation_count") / col("total_count")) * 100)
print(joined_counts.show(truncate=False))
# Filter labcodes where null interpretation percentage is greater than 90%
filtered_labcodes = joined_counts.filter(col("null_interpretation_percentage") > 90)

# Count the number of such unique labcodes
count_filtered_labcodes = filtered_labcodes.count()

# Display the count of unique labcodes
print("Number of unique labcodes with more than 90% null New_updated_Interpretation:", count_filtered_labcodes)

+---------+-----------+-------------------------+------------------------------+
|labcode  |total_count|null_interpretation_count|null_interpretation_percentage|
+---------+-----------+-------------------------+------------------------------+
|10706-0  |4          |4                        |100.0                         |
|11274-8  |1543       |315                      |20.414776409591703            |
|11556-8  |2025       |537                      |26.51851851851852             |
|121870001|215        |215                      |100.0                         |
|122296009|74         |74                       |100.0                         |
|12438-8  |1          |0                        |0.0                           |
|15586-1  |35         |29                       |82.85714285714286             |
|15759-4  |1          |1                        |100.0                         |
|16052-3  |217        |207                      |95.39170506912443             |
|16276-8  |5          |5    

In [7]:
Final_Cohort_Lab_toStack.printSchema()
Final_Control_Lab_toStack.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- labcode: string (nullable = true)
 |-- New_updated_Interpretation: string (nullable = true)
 |-- servicedate: string (nullable = true)

